In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely.affinity import translate
from scipy.spatial.distance import pdist, squareform
from skimage.draw import polygon
from scipy.signal import fftconvolve

In [ ]:
import csv
import pickle
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
TARGET_SCALE = "min_234"
MIN_CELL_AREA = 10000

DATASET_CONFIGS = {
    1: {
        "image_path": "Lysosome_masked_crops_1024.tif",
        "csv_path": "PhTau_histogram_clusters.csv",
        "coord_path": "processed_data/spot_coordinates.pkl",
        "delaunay_path": "processed_data/delaunay_results.pkl",
    },
    2: {
        "image_path": "cell_gallery_1024x1024_6ch.tif",
        "csv_path": "gallery_phtau_ground_truth.csv",
        "coord_path": "processed_data_2/spot_coordinates.pkl",
        "delaunay_path": "processed_data_2/delaunay_results.pkl",
    },
}

# Master data containers
images = {}
n_images = {}
labels = {}
categories = {}
coord_map = {}
delaunay_objects_dict = {}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_csv_labels(csv_path, total_expected_frames):
    """Parses cluster labels from CSV and pads remaining frames as untreated."""
    parsed_labels = []
    with open(csv_path, "r") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header row
        for row in reader:
            val = row[-1]
            if val == "0":
                parsed_labels.append(["0", "dox+, PhTau+"])
            elif val == "1":
                parsed_labels.append(["1", "dox+, PhTau-"])

    # Pad missing indices up to the total stack size (assumed untreated)
    while len(parsed_labels) < total_expected_frames:
        parsed_labels.append(["2", "dox-"])
        
    return parsed_labels


def load_pickle_scale(pickle_path, scale_key):
    """Helper to extract a specific resolution scale from a pickle file."""
    with open(pickle_path, "rb") as f:
        return pickle.load(f)[scale_key]


def compute_valid_mask_indices(image_stack, area_threshold):
    """ Vectorised masking and area filter over a standardised image stack."""
    # Since channel is guaranteed to be at axis -1, collapse it if 4D
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0

    # Fill internal segmentation holes slice-by-slice across the stack
    cell_masks = binary_fill_holes(raw_masks)
    
    # Calculate pixel surface areas by summing spatial dimensions (axes 1 and 2)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    
    # Return boolean array masking frames that meet size criteria
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
for idx, config in DATASET_CONFIGS.items():
    # Load image stack pipeline
    raw_images_stack = io.imread(config["image_path"])

    # Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
    if raw_images_stack.ndim == 4:
        # If your channel is currently at axis 1 (e.g., shape 100, 6, 1024, 1024)
        channel_axis = np.argmin(raw_images_stack.shape)
        if channel_axis != -1:
            raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)
    
    total_frames = raw_images_stack.shape[0]

    # Load and map structural metadata sequentially
    all_labels = load_csv_labels(config["csv_path"], total_frames)
    full_coord = load_pickle_scale(config["coord_path"], TARGET_SCALE)
    full_delaunay = load_pickle_scale(config["delaunay_path"], TARGET_SCALE)

    # Compute valid mask filter indices using vectorised area sums
    valid_indices = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

    # Apply boolean index filtering to numerical arrays
    images[idx] = raw_images_stack[valid_indices]
    n_images[idx] = np.sum(valid_indices)
    
    # Filter Python list metadata using matching list comprehensions
    labels[idx] = [all_labels[i] for i, valid in enumerate(valid_indices) if valid]
    coord_map[idx] = [full_coord[i] for i, valid in enumerate(valid_indices) if valid]
    delaunay_objects_dict[idx] = [full_delaunay[i] for i, valid in enumerate(valid_indices) if valid]

    # Parse remaining categories, sorting by category id descending
    unique_pairs = {tuple(item) for item in labels[idx]}
    sorted_pairs = sorted(unique_pairs, key=lambda x: int(x[0]), reverse=True)
    categories[idx] = [pair[1] for pair in sorted_pairs]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial import distance

# 1. Structure data collection by dataset and combined
raw_distances = {1: {}, 2: {}, "Combined": {}}
# Clearly defining Conditions/conditions instead of time-points
categories_list = ["dox+, PhTau+", "dox+, PhTau-", "dox-"]

for d_id in raw_distances:
    for cat in categories_list:
        raw_distances[d_id][cat] = []

# 2. Extract nearest-neighbour distances per cell
for dataset_idx, dataset_coords in coord_map.items():
    dataset_labels = labels[dataset_idx]

    # Iterating over each cell's lysosome coordinate list
    for img_idx, coord_list in enumerate(dataset_coords):
        label_string = dataset_labels[img_idx][1]  # Condition string
        coords = np.array(coord_list)

        # Distance calculation requires at least two points per cell
        if len(coords) > 1:
            dist_matrix = distance.cdist(coords, coords)
            np.fill_diagonal(dist_matrix, np.inf)
            nn_distances = np.min(dist_matrix, axis=1).tolist()

            # Append to specific dataset and combined pool
            raw_distances[dataset_idx][label_string].extend(nn_distances)
            raw_distances["Combined"][label_string].extend(nn_distances)

# 3. Convert collected lists into a tidy pandas DataFrame
plot_data = []
for dataset_id, cats in raw_distances.items():
    for cat_name, dist_list in cats.items():
        for dist in dist_list:
            plot_data.append(
                {
                    "Dataset": (
                        f"Dataset {dataset_id}"
                        if isinstance(dataset_id, int)
                        else dataset_id
                    ),
                    "Condition": cat_name,
                    "Distance": dist * 0.0739859,  # Convert pixels to µm
                }
            )

df = pd.DataFrame(plot_data)

# Global condition mapping order for disease progression
condition_order = ["dox-", "dox+, PhTau-", "dox+, PhTau+"]
df["Condition"] = pd.Categorical(
    df["Condition"], categories=condition_order, ordered=True
)

# 4. Generate the 3-panel figure
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Create output folder
output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

condition_order = ["dox-", "dox+, PhTau-", "dox+, PhTau+"]
palette = {
    "dox-": "green",
    "dox+, PhTau-": "orange",
    "dox+, PhTau+": "red",
}


def add_kde_plot(data, dataset_name, ax, title=True):
    """Add one nearest-neighbour KDE plot to an existing axis."""

    sub_df = data[data["Dataset"] == dataset_name].copy()
    sub_df = sub_df.dropna(subset=["Distance"])

    ax.set_xlim(0, 7)

    if not sub_df.empty:
        active_hue_order = [
            condition
            for condition in condition_order
            if condition in sub_df["Condition"].astype(str).unique()
        ]

        if active_hue_order:
            sns.kdeplot(
                data=sub_df,
                x="Distance",
                hue="Condition",
                hue_order=active_hue_order,
                palette=palette,
                common_norm=False,
                linewidth=2.5,
                warn_singular=False,
                ax=ax,
                legend=False
            )
        else:
            ax.text(
                0.5,
                0.5,
                "No disease-state data available",
                ha="center",
                va="center",
                color="gray",
                transform=ax.transAxes,
            )
    else:
        ax.text(
            0.5,
            0.5,
            "No Data Available",
            ha="center",
            va="center",
            color="gray",
            transform=ax.transAxes,
        )

    if title==True:
        ax.set_title(dataset_name, fontsize=14, fontweight="bold")
    ax.set_xlabel("Nearest Neighbour Distance (µm)", fontsize=12)
    ax.set_ylabel("Probability Density", fontsize=12)


# ------------------------------------------------------------------
# Dataset 1 and Dataset 2 side-by-side
# ------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharex=True,
    sharey=True,
)

add_kde_plot(df, "Dataset 1", axes[0])
add_kde_plot(df, "Dataset 2", axes[1])

fig.savefig(
    output_dir / "nearest_neighbour_dataset_1_and_2.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------------
# Combined dataset as a separate plot
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 3))

add_kde_plot(df, "Combined", ax, title=False)

fig.savefig(
    output_dir / "nearest_neighbour_combined.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import distance_transform_edt

# --- CONFIGURATION & STORAGE ---
CELL_CHANNEL_IDX = 4
NUCL_CHANNEL_IDX = 5  
DS_IDX = 2
CONDITION_ORDER = ["dox+, PhTau-", "dox+, PhTau+"]
PALETTE = {"dox+, PhTau-": "orange", "dox+, PhTau+": "red"}

# Updated: Unified results container mapped to the exact protocol
results = {cond: [] for cond in CONDITION_ORDER}

# --- DATA PROCESSING ---
if DS_IDX in images:
    for idx, (label_id, label_text) in enumerate(labels[DS_IDX]):
        if label_text not in CONDITION_ORDER:
            continue
            
        img = images[DS_IDX][idx]
        
        # 1. Extract Masks safely based on image dimensions
        if img.ndim == 4 and img.shape[1] == 6:
            nuc_mask = img[NUCL_CHANNEL_IDX, :, :] > 0  
            cell_mask = img[CELL_CHANNEL_IDX, :, :] > 0
        else:
            nuc_mask = img[:, :, NUCL_CHANNEL_IDX] > 0  
            cell_mask = img[:, :, CELL_CHANNEL_IDX] > 0
            
        # Define valid lysosome spatial masking boundaries
        nuc_pixels = np.column_stack(np.nonzero(nuc_mask))
        if len(nuc_pixels) == 0:
            continue  
            
        combined_coords = np.array(coord_map[DS_IDX][idx])
        if len(combined_coords) == 0:
            continue
            
        lyso_y = np.clip(np.round(combined_coords[:, 0]).astype(int), 0, nuc_mask.shape[0] - 1)
        lyso_x = np.clip(np.round(combined_coords[:, 1]).astype(int), 0, nuc_mask.shape[1] - 1)
        
        # Only analyse lysosomes that do not overlap with the nucleus mask
        valid_lyso = ~nuc_mask[lyso_y, lyso_x]
        if np.any(valid_lyso):
            valid_y = lyso_y[valid_lyso]
            valid_x = lyso_x[valid_lyso]
            
            # Step 1: Distance transform to calculate distance to the nearest nucleus boundary point
            dist_to_nuc_edge_map = distance_transform_edt(~nuc_mask)
            
            # Step 2: Extract individual raw distances: D_TN(i)
            raw_dist_to_nuc = dist_to_nuc_edge_map[valid_y, valid_x]
            
            # Step 3: Extract maximum to-nucleus distance for this specific cell: D_TN^Max
            d_tn_max = np.max(raw_dist_to_nuc)
            
            if d_tn_max > 0:
                # Step 4: Normalize: D_TN(i) / D_TN^Max
                protocol_norm = raw_dist_to_nuc / d_tn_max
                results[label_text].extend(protocol_norm)

# --- VISUALISATION PREPARATION ---
def build_dataframe(results_dict):
    """Converts dictionary results securely into a tidy DataFrame format."""
    plot_data = [
        {"Condition": cat, "Normalized_Distance": dist}
        for cat, dist_array in results_dict.items() for dist in dist_array
    ]
    df = pd.DataFrame(plot_data)
    if not df.empty:
        df["Condition"] = pd.Categorical(df["Condition"], categories=CONDITION_ORDER, ordered=True)
    return df

# --- PLOTTING GENERATION ---
from pathlib import Path

# Output folder
output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

# --- PLOTTING GENERATION ---
sns.set_theme(style="whitegrid")
df_plot = build_dataframe(results)

# Match the nearest-neighbour plot dimensions
fig, ax = plt.subplots(figsize=(5, 3))

if not df_plot.empty and df_plot["Normalized_Distance"].notna().sum() > 0:
    df_clean = df_plot.dropna(subset=["Normalized_Distance"]).copy()

    df_clean["Condition"] = (
        df_clean["Condition"].cat.remove_unused_categories()
    )

    active_hue = [
        condition
        for condition in CONDITION_ORDER
        if condition in df_clean["Condition"].astype(str).unique()
    ]

    sns.kdeplot(
        data=df_clean,
        x="Normalized_Distance",
        hue="Condition",
        hue_order=active_hue,
        palette=PALETTE,
        ax=ax,
        common_norm=False,
        linewidth=2.5,
        warn_singular=False,
        clip=(0.0, 1.0),
        legend=False
    )

    ax.set_xlim(0.0, 1.0)

else:
    ax.text(
        0.5,
        0.5,
        "No Data Available",
        ha="center",
        va="center",
        color="gray",
        transform=ax.transAxes,
    )

# No title
ax.set_title("")

# Match the nearest-neighbour axis-label styling
ax.set_xlabel("Normalised To-Nucleus Distance", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)

# Save as SVG
fig.savefig(
    output_dir / f"normalised_to_nucleus_distance_dataset_{DS_IDX}.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial import distance

# 1. Structure data collection by dataset and combined
inter_organelle_results = {1: {}, 2: {}, "Combined": {}}
categories_list = ["dox+, PhTau+", "dox+, PhTau-", "dox-"]

for d_id in inter_organelle_results:
    for cat in categories_list:
        inter_organelle_results[d_id][cat] = []

# Conversion factor: pixels to µm
PIXEL_TO_UM = 0.0739859

# 2. Extract normalised pairwise inter-organelle distances per cell
for dataset_idx, dataset_coords in coord_map.items():
    dataset_labels = labels[dataset_idx]

    # Iterating over each cell's organelle coordinate list
    for img_idx, coord_list in enumerate(dataset_coords):
        label_string = dataset_labels[img_idx][1]  # Condition string
        
        # Extract coordinates and immediately convert from pixels to µm
        combined = np.array(coord_list) * PIXEL_TO_UM
        second = np.array(coord_list) * PIXEL_TO_UM
        
        if len(combined) == 0 or len(second) < 2:
            continue
            
        # Establish cell centre dynamically using Organelle Geometric Centre
        cell_center = np.mean(combined, axis=0).reshape(1, 2)
        
        # Calculate cell radius metric based on the determined centre
        cell_radius = np.max(distance.cdist(cell_center, combined))
        if cell_radius == 0:
            cell_radius = 1.0
            
        # Compute unique pairwise distances between all lysosomes
        second_dist_matrix = distance.cdist(second, second)
        triu_indices = np.triu_indices(len(second), k=1)
        inter_org_raw = second_dist_matrix[triu_indices]
        
        if len(inter_org_raw) == 0:
            continue
        
        # Find the maximum pairwise inter-organelle distance
        d_io_max = np.max(inter_org_raw)
        if d_io_max == 0:
            continue
        
        # Normalise by D_IO_Max
        normalized_distances = (inter_org_raw / d_io_max).tolist()
        
        # Append to specific dataset and combined pool
        inter_organelle_results[dataset_idx][label_string].extend(normalized_distances)
        inter_organelle_results["Combined"][label_string].extend(normalized_distances)

# 3. Convert collected lists into a tidy pandas DataFrame
plot_data = []
for dataset_id, cats in inter_organelle_results.items():
    for cat_name, dist_list in cats.items():
        for dist in dist_list:
            plot_data.append(
                {
                    "Dataset": (
                        f"Dataset {dataset_id}"
                        if isinstance(dataset_id, int)
                        else dataset_id
                    ),
                    "Condition": cat_name,
                    "Normalized_Distance": dist,
                }
            )

df = pd.DataFrame(plot_data)

# Global condition mapping order for disease progression
condition_order = ["dox-", "dox+, PhTau-", "dox+, PhTau+"]
df["Condition"] = pd.Categorical(
    df["Condition"], categories=condition_order, ordered=True
)

from pathlib import Path

# --- 4. PLOTTING AND SVG EXPORT ---

output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

palette = {
    "dox-": "green",
    "dox+, PhTau-": "orange",
    "dox+, PhTau+": "red",
}

MAX_X_LIMIT = 1.0


def add_kde_plot(data, dataset_name, ax, title=True):
    """Draw a normalized inter-organelle distance KDE on a supplied axis."""

    sub_df = data[data["Dataset"] == dataset_name].dropna(
        subset=["Normalized_Distance"]
    ).copy()

    ax.set_xlim(0.0, MAX_X_LIMIT)

    if not sub_df.empty:
        sub_df["Condition"] = (
            sub_df["Condition"].cat.remove_unused_categories()
        )

        active_hue_order = [
            condition
            for condition in condition_order
            if condition in sub_df["Condition"].astype(str).unique()
        ]

        if active_hue_order:
            sns.kdeplot(
                data=sub_df,
                x="Normalized_Distance",
                hue="Condition",
                hue_order=active_hue_order,
                palette=palette,
                ax=ax,
                common_norm=False,
                linewidth=2.5,
                warn_singular=False,
                clip=(0.0, MAX_X_LIMIT),
                legend=False
            )
        else:
            ax.text(
                0.5,
                0.5,
                "No disease-state data available",
                ha="center",
                va="center",
                color="gray",
                transform=ax.transAxes,
            )
    else:
        ax.text(
            0.5,
            0.5,
            "No Data Available",
            ha="center",
            va="center",
            color="gray",
            transform=ax.transAxes,
        )

    if title:
        ax.set_title(dataset_name, fontsize=14, fontweight="bold")

    ax.set_xlabel("Normalised Inter-Organelle Distance", fontsize=12)
    ax.set_ylabel("Probability Density", fontsize=12)


# ------------------------------------------------------------------
# Dataset 1 and Dataset 2: side-by-side figure
# ------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharex=True,
    sharey=True,
)

add_kde_plot(df, "Dataset 1", axes[0])
add_kde_plot(df, "Dataset 2", axes[1])

fig.savefig(
    output_dir / "inter_organelle_dataset_1_and_2.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------------
# Combined data: separate figure with no title
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 3))

add_kde_plot(df, "Combined", ax, title=False)

fig.savefig(
    output_dir / "inter_organelle_combined.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
import csv
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
CLUSTER_CSV = "Lysosome_annotations_combined.csv"
IMAGE_PATH = "Lysosome_masked_crops_1024.tif"  # Or your corresponding tif file
MIN_CELL_AREA = 10000

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def compute_valid_mask_indices(image_stack, area_threshold):
    """Vectorised masking and area filter over a standardised image stack."""
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0
    cell_masks = binary_fill_holes(raw_masks)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE SYNCHRONIZATION
# ==========================================
# 1. Load the original image stack to deduce which frame indices survived filtering
raw_images_stack = io.imread(IMAGE_PATH)

# Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
if raw_images_stack.ndim == 4:
    channel_axis = np.argmin(raw_images_stack.shape)
    if channel_axis != -1:
        raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)

# 2. Generate the exact boolean mask array used by your image data pipeline
# True means the cell survived, False means it was dropped due to size
survived_import_mask = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

# ==========================================
# 4. PARSE & FILTER ANNOTATIONS
# ==========================================
annotations = []
valid_indices = []

with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    
    for i, row in enumerate(reader):
        # A. FIRST CHECK: Did this cell index survive the image import drop step?
        # If the image index is out of bounds or marked False in the mask, skip it.
        if i >= len(survived_import_mask) or not survived_import_mask[i]:
            continue

        # B. SECOND CHECK: Filter out "NoClass" (annotation -1.0)
        if row['annotation_names'] == "NoClass" or float(row['annotations']) == -1.0:
            continue
            
        # C. Extract relevant data
        ann_name = row['annotation_names']
        phtau_val = row['PhTau_cluster']
        treatment_val = row['treatment']
        
        # D. Create descriptive label
        phtau_status = "PhTau+" if phtau_val == '1' else "PhTau-"
        full_label = f"{ann_name}, {phtau_status}, {treatment_val}"
        
        # Store label and map its strict index configuration
        annotations.append([row['annotations'], full_label])
        valid_indices.append(i)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial import distance
from scipy.stats import gaussian_kde

# 1. Isolate clean structural names from your pre-imported annotations list
# Safely extracts just the name (e.g., "normal") by taking the string before the comma
clean_categories = []
for row in annotations:
    # row looks like ['-1.0', 'normal, PhTau-, dox-'] or similar based on your extraction code
    full_label = row[1]
    base_name = full_label.split(",")[0].strip()
    clean_categories.append(base_name)

unique_categories = list(set(clean_categories))
dataset1_distances = {cat: [] for cat in unique_categories}

# 2. Extract distances for Dataset 1 using your valid_indices map
dataset_coords = coord_map[1]

for img_idx, coord_list in enumerate(dataset_coords):
    # Only process if this image index survived your CSV filtering criteria
    if img_idx in valid_indices:
        # Find its sequential position inside your pre-imported annotations list
        list_pos = valid_indices.index(img_idx)
        cat_name = clean_categories[list_pos]

        coords = np.array(coord_list)
        if len(coords) > 1:
            dist_matrix = distance.cdist(coords, coords)
            np.fill_diagonal(dist_matrix, np.inf)
            nn_distances = np.min(dist_matrix, axis=1).tolist()

            dataset1_distances[cat_name].extend(nn_distances)

# 3. Convert collected lists into a tidy pandas DataFrame
plot_data_d1 = []
for cat_name, dist_list in dataset1_distances.items():
    for dist in dist_list:
        plot_data_d1.append(
            {
                "Structural Category": cat_name,
                "Distance": dist * 0.0739859,  # Convert pixels to µm
            }
        )

df_d1 = pd.DataFrame(plot_data_d1)

# 4. Calculate empirical peak heights to determine legend order
evaluation_points = np.linspace(0, 7, 500)
peak_heights = {}

for cat in df_d1["Structural Category"].unique():
    subset = df_d1[df_d1["Structural Category"] == cat]["Distance"].dropna()
    if len(subset) > 1:
        # Evaluate density over the same window to mathematically identify the highest peak
        kde_func = gaussian_kde(subset)
        densities = kde_func(evaluation_points)
        peak_heights[cat] = np.max(densities)
    else:
        peak_heights[cat] = 0

# Sort categories from tallest peak density to shortest
sorted_by_peak = sorted(peak_heights, key=peak_heights.get, reverse=True)

# 5. Generate the final single-panel continuous density plot

# Explicit annotation order
annotation_order = [
    "normal",
    "enlarged",
    "dispersed",
    "enlarged&dispersed",
    "big clusters",
    "other/atypical",
]

annotation_colours = {
    "big clusters": "#F1C40F",
    "dispersed": "#3498DB",
    "enlarged": "#E74C3C",
    "enlarged&dispersed": "#9B59B6",
    "normal": "#2ECC71",
    "other/atypical": "#95A5A6",
}

# Retain only annotations present in Dataset 1
active_annotation_order = [
    annotation
    for annotation in annotation_order
    if annotation in df_d1["Structural Category"].unique()
]

# Generate the final single-panel continuous density plot
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(left=0, right=7)

if not df_d1.empty and df_d1["Distance"].notna().sum() > 0:
    sns.kdeplot(
        data=df_d1,
        x="Distance",
        hue="Structural Category",
        hue_order=active_annotation_order,
        palette=annotation_colours,
        ax=ax,
        common_norm=False,
        linewidth=2.5,
        warn_singular=False,
        legend=False
    )

    # Change the legend title
    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Annotation")

else:
    ax.text(
        0.5,
        0.5,
        "No Data Available",
        ha="center",
        va="center",
        color="gray",
        transform=ax.transAxes,
    )

# ax.set_title(
#     "Lysosome Spacing by Annotation",
#     fontsize=14,
#     fontweight="bold",
# )

ax.set_xlabel("Nearest Neighbour Distance (µm)", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)

# Create output directory
output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the plot
fig.savefig(
    output_dir / "nearest_neighbour_by_annotation.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy.spatial import distance
from scipy.stats import gaussian_kde

# Conversion factor: pixels to µm
PIXEL_TO_UM = 0.0739859
MAX_X_LIMIT = 1.0

# Explicit annotation order and consistent colour mapping
annotation_order = [
    "normal",
    "enlarged",
    "dispersed",
    "enlarged&dispersed",
    "big clusters",
    "other/atypical",
]

annotation_colours = {
    "big clusters": "#F1C40F",
    "dispersed": "#3498DB",
    "enlarged": "#E74C3C",
    "enlarged&dispersed": "#9B59B6",
    "normal": "#2ECC71",
    "other/atypical": "#95A5A6",
}

# 1. Extract clean structural annotation names
clean_categories = []

for row in annotations:
    full_label = row[1]
    base_name = full_label.split(",")[0].strip()
    clean_categories.append(base_name)

unique_categories = list(set(clean_categories))
dataset1_inter_distances = {category: [] for category in unique_categories}

# 2. Extract normalised pairwise distances for Dataset 1
dataset_coords = coord_map[1]

for img_idx, coord_list in enumerate(dataset_coords):

    # Process only images retained after CSV filtering
    if img_idx not in valid_indices:
        continue

    list_pos = valid_indices.index(img_idx)
    category = clean_categories[list_pos]

    # Convert coordinates from pixels to µm
    coords = np.asarray(coord_list, dtype=float) * PIXEL_TO_UM

    # At least two organelles are required
    if len(coords) < 2:
        continue

    # Calculate unique inter-organelle distances
    dist_matrix = distance.cdist(coords, coords)
    upper_triangle_indices = np.triu_indices(len(coords), k=1)
    inter_organelle_distances = dist_matrix[upper_triangle_indices]

    if len(inter_organelle_distances) == 0:
        continue

    # Normalise by the maximum distance within each cell
    maximum_distance = np.max(inter_organelle_distances)

    if maximum_distance == 0:
        continue

    normalised_distances = inter_organelle_distances / maximum_distance
    dataset1_inter_distances[category].extend(normalised_distances.tolist())

# 3. Convert to a tidy DataFrame
plot_data_inter = [
    {
        "Structural Category": category,
        "Normalized_Distance": normalised_distance,
    }
    for category, distance_list in dataset1_inter_distances.items()
    for normalised_distance in distance_list
]

df_inter = pd.DataFrame(
    plot_data_inter,
    columns=["Structural Category", "Normalized_Distance"],
)

# 4. Retain only annotations present in Dataset 1
if not df_inter.empty:
    present_categories = set(df_inter["Structural Category"].unique())

    active_annotation_order = [
        annotation
        for annotation in annotation_order
        if annotation in present_categories
    ]
else:
    active_annotation_order = []

# 5. Generate the single-panel KDE plot
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(left=0, right=MAX_X_LIMIT)

if (
    not df_inter.empty
    and df_inter["Normalized_Distance"].notna().sum() > 0
    and active_annotation_order
):
    sns.kdeplot(
        data=df_inter,
        x="Normalized_Distance",
        hue="Structural Category",
        hue_order=active_annotation_order,
        palette=annotation_colours,
        common_norm=False,
        linewidth=2.5,
        clip=(0.0, MAX_X_LIMIT),
        warn_singular=False,
        ax=ax,
        legend=False
    )

    # Match the annotation plot legend styling
    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Annotation")

else:
    ax.text(
        0.5,
        0.5,
        "No Data Available",
        ha="center",
        va="center",
        color="gray",
        transform=ax.transAxes,
    )

# Consistent axis styling
ax.set_xlabel("Normalised Inter-Organelle Distance", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)

# Save as SVG in the Figures directory
output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    output_dir / "inter_organelle_by_annotation.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)